In [ ]:
import nest_asyncio, threading, uvicorn, os, json, datetime, io, socket, struct, time, asyncio, requests
import numpy as np
from PIL import Image, ImageChops, ImageEnhance
from fastapi import FastAPI, Request, UploadFile, File, Form
from fastapi.responses import HTMLResponse, RedirectResponse
from fastapi.staticfiles import StaticFiles
from collections import defaultdict

nest_asyncio.apply()
if not os.path.exists("static"): os.makedirs("static")

LOG_FILE = "phantasm_forensic.log"
DISCORD_WEBHOOK_URL = "https://discord.com/api/webhooks/1496061853464531004/3ICsYjH8QevdXMsW3NsHp-Zc5lF-u7x9ixkngP_jN0f0ya2Y9ZfSRWoYnmmK0L6e4JWJ" # Optional: Paste Discord URL here

def log_event(event_type, ip, details):
    timestamp = datetime.datetime.now().isoformat()
    entry = {"timestamp": timestamp, "event_type": event_type, "ip": ip, "details": details}
    with open(LOG_FILE, "a") as f:
        f.write(json.dumps(entry) + "\n")

    if event_type == "LOGIN_ATTEMPT" and DISCORD_WEBHOOK_URL.startswith("http"):
        verdict = details.get('verdict', '???')
        color = 0xFF4444 if verdict == "AI/SYNTHETIC" else 0x44FF44
        payload = {"embeds": [{"title": "🛡️ Phantasm Alert", "color": color, "fields": [
            {"name": "Verdict", "value": verdict, "inline": True},
            {"name": "User", "value": details.get('attempted_username', 'Guest'), "inline": True},
            {"name": "IP", "value": ip, "inline": True}]}]}
        threading.Thread(target=lambda: requests.post(DISCORD_WEBHOOK_URL, json=payload)).start()

app = FastAPI()
app.mount("/static", StaticFiles(directory="static"), name="static")
print("✅ Cell 1: Core Initialized")

In [ ]:
def run_ela(image_bytes: bytes):
    # Detects if pixels have been resaved or manipulated
    original = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    temp = io.BytesIO()
    original.save(temp, format="JPEG", quality=90)
    resaved = Image.open(temp)

    ela = ImageChops.difference(original, resaved)
    extrema = ela.getextrema()
    max_diff = max([ex[1] for ex in extrema]) or 1
    ela = ImageEnhance.Brightness(ela).enhance(255.0 / max_diff)
    ela.save("static/ela_output.jpg")

    score = np.mean(np.array(ela))
    prob = min(score / 30.0, 1.0)
    return {
        "ai_probability": round(prob * 100, 2),
        "verdict": "AI/SYNTHETIC" if prob > 0.55 else "LIKELY REAL"
    }
print("✅ Cell 2: AI Engine Ready")

In [ ]:
@app.get("/", response_class=HTMLResponse)
async def login(request: Request):
    return """<body style="background:#050505; color:#00e5ff; font-family:monospace; display:flex; justify-content:center; align-items:center; height:100vh; margin:0;">
    <div style="border:1px solid #00e5ff; padding:30px; background:rgba(0,20,20,0.9); width:400px; box-shadow:0 0 20px #00e5ff44;">
        <h2 style="text-align:center; letter-spacing:3px;">SECURE GATEWAY</h2>
        <form onsubmit="event.preventDefault(); const d=new FormData(); d.append('file', document.getElementById('i').files[0]); d.append('u', document.getElementById('un').value); fetch('/analyze', {method:'POST', body:d}).then(()=>window.location.href='/dashboard');">
            <input type="text" id="un" placeholder="OPERATOR ID" style="width:100%; background:#000; color:#00e5ff; border:1px solid #0081a7; padding:10px; margin:10px 0;" required>
            <input type="password" placeholder="ACCESS KEY" style="width:100%; background:#000; color:#00e5ff; border:1px solid #0081a7; padding:10px; margin:10px 0;" required>
            <input type="file" id="i" style="width:100%; background:#000; color:#00e5ff; border:1px solid #0081a7; padding:10px; margin:10px 0;" required>
            <button type="submit" style="width:100%; background:#00e5ff; color:#000; border:none; padding:12px; font-weight:bold; cursor:pointer;">INITIALIZE HANDSHAKE</button>
        </form>
    </div></body>"""

@app.post("/analyze")
async def analyze(request: Request, file: UploadFile = File(...), u: str = Form("Guest")):
    res = run_ela(await file.read())
    res["attempted_username"] = u
    log_event("LOGIN_ATTEMPT", request.client.host, res)
    return {"status": "verified"}

@app.get("/dashboard", response_class=HTMLResponse)
async def dash():
    return """<body style="background:#0a0a0a; color:#00ff41; font-family:monospace; padding:40px;">
    <div style="border:1px solid #333; padding:20px; max-width:600px; margin:auto; background:#000;">
        <h1 style="border-bottom:2px solid #00ff41;">INTERNAL DATA REPOSITORY</h1>
        <p>📄 financial_data_2026.pdf <button onclick="alert('Permission Denied')">DOWNLOAD</button></p>
        <p>🗄️ customer_db.sql <button onclick="alert('SSH Key Required')">DOWNLOAD</button></p>
    </div></body>"""

@app.get("/admin/clear")
async def clear_logs_route():
    if os.path.exists(LOG_FILE):
        open(LOG_FILE, 'w').close() # Truncate instead of delete to avoid file lock issues
    if os.path.exists("static/ela_output.jpg"):
        try: os.remove("static/ela_output.jpg")
        except: pass
    return RedirectResponse(url="/admin/phantasm-secret-42")

@app.get("/admin/phantasm-secret-42", response_class=HTMLResponse)
async def admin_panel():
    logs = []
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, "r") as f:
            for line in f:
                try: logs.append(json.loads(line))
                except: continue
    ai_count = sum(1 for e in logs if e['details'].get('verdict') == "AI/SYNTHETIC")
    rows = "".join([f"<tr style='border-bottom:1px solid #333;'><td>{e['timestamp'][11:19]}</td><td>{e['ip']}</td><td>{e['details'].get('attempted_username','Guest')}</td><td style='color:{'#ff4444' if e['details'].get('verdict')=='AI/SYNTHETIC' else '#44ff44'}'>{e['details'].get('verdict')}</td></tr>" for e in reversed(logs[-20:])])
    return f"""<html><head><meta http-equiv="refresh" content="5"><style>body{{background:#111;color:#eee;font-family:sans-serif;padding:20px;}} .card{{background:#222;padding:15px;border-left:4px solid #00e5ff;margin:10px;flex:1;}}</style></head><body>
    <div style="display:flex; justify-content:space-between; border-bottom:2px solid #00e5ff;"><h1>PHANTASM CONSOLE v4.2</h1><a href="/admin/clear" style="color:#ff4444; text-decoration:none; font-weight:bold; padding-top:20px;">PURGE SYSTEM LOGS</a></div>
    <div style="display:flex;"><div class="card">TOTAL EVENTS<br><span style="font-size:24px;">{len(logs)}</span></div><div class="card">AI THREATS<br><span style="font-size:24px; color:#ff4444;">{ai_count}</span></div></div>
    <div style="background:#000; padding:20px; text-align:center; border:1px solid #333; margin:20px 0;"><h3>LATEST FORENSIC ELA VIEW</h3>
    {"<img src='/static/ela_output.jpg' style='max-width:300px; border:1px solid #00e5ff;'>" if os.path.exists("static/ela_output.jpg") else "<p>No active forensics.</p>"}</div>
    <table border="0" style="width:100%; border-collapse:collapse; text-align:left;"><tr><th>TIME</th><th>IP</th><th>USER</th><th>VERDICT</th></tr>{rows}</table></body></html>"""
print("✅ Cell 3: Interface Deployed")

In [ ]:
def start_sniffer():
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_RAW, socket.IPPROTO_IP)
        s.bind((socket.gethostbyname(socket.gethostname()), 0))
        s.setsockopt(socket.IPPROTO_IP, socket.IP_HDRINCL, 1)
        s.ioctl(socket.SIO_RCVALL, socket.RCVALL_ON)
        print("🛡️ Sniffer Active...")
        while True:
            data, _ = s.recvfrom(65535)
            if struct.unpack('!BBHHHBBH4s4s', data[0:20])[6] == 6 and (struct.unpack('!HHLLBBHHH', data[20:40])[5] & 0x02):
                print("\n🚨 ALERT: Port Scan detected!")
    except: print("❌ Sniffer: Admin rights missing.")

# Launch Server
threading.Thread(target=start_sniffer, daemon=True).start()
loop = asyncio.get_event_loop()
loop.create_task(uvicorn.Server(uvicorn.Config(app=app, host="0.0.0.0", port=8002)).serve())

local_ip = socket.gethostbyname(socket.gethostname())
print(f"\n✨ PROJECT PHANTASM LIVE ✨")
print(f"Attacker (Handheld Device): http://{local_ip}:8002")
print(f"Admin (Control Panel): http://localhost:8002/admin/phantasm-secret-42")